In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("../src")

from sentence_transformers import SentenceTransformer
from data_loader import load_candidates
from job_representation import build_job_description
from retrieval import Retriever
from ranking import Ranker

# Load model only once
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

retriever = Retriever(embedding_model)
ranker = Ranker(embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from data_loader import load_candidates

candidates = load_candidates(
    "../data/candidates.jsonl",
    limit=100
)


In [4]:
from docx import Document
from job_representation import build_job_description

doc = Document("../India_Runs_Hackathon/job_description.docx")

jd_text = "\n".join(
    para.text
    for para in doc.paragraphs
)

job = build_job_description(jd_text)

In [5]:
ranker.build_requirement_embeddings(job)

ranker.build_experience_embeddings(candidates)

ranker.build_document_embeddings(
    candidates
)

In [6]:
rankings = ranker.rank_candidates(
    job,
    candidates
)

retrieval systems         Weight=3.0 Best=0.230
ranking systems           Weight=3.0 Best=0.115
recommendation systems    Weight=3.0 Best=0.126
search systems            Weight=3.0 Best=0.231
embeddings                Weight=2.5 Best=0.054
vector databases          Weight=2.5 Best=0.256
python                    Weight=2.0 Best=0.129
llms                      Weight=1.5 Best=0.074
fine-tuning               Weight=1.0 Best=0.227
bm25                      Weight=2.0 Best=0.087

Final Evidence Score = 0.155
retrieval systems         Weight=3.0 Best=0.210
ranking systems           Weight=3.0 Best=0.266
recommendation systems    Weight=3.0 Best=0.203
search systems            Weight=3.0 Best=0.315
embeddings                Weight=2.5 Best=0.060
vector databases          Weight=2.5 Best=0.200
python                    Weight=2.0 Best=0.326
llms                      Weight=1.5 Best=0.310
fine-tuning               Weight=1.0 Best=0.220
bm25                      Weight=2.0 Best=0.263

Final Evi

In [7]:
candidate = rankings[0]["candidate"]

for req, info in candidate.matched_experiences.items():
    print("=" * 50)
    print(req)
    print("Score:", round(info["score"], 3))

    if info["experience"]:
        print("Title:", info["experience"].title)
        print(info["experience"].description[:200])

retrieval systems
Score: 0.371
Title: Search Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
ranking systems
Score: 0.45
Title: Recommendation Systems Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
recommendation systems
Score: 0.53
Title: Recommendation Systems Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
search systems
Score: 0.418
Title: Search Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: conte

In [12]:
print(candidates[0].retrieval_document)

Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.
Headline: Backend Engineer | SQL, Spark, Cloud
Skills: Tailwind, NLP, Image Classification, Fine-tuning LLMs, Weights & Biases, Speech Recognition, Photoshop, TTS, LoRA, Apache Beam, AWS, Flask, BentoML, Milvus, GANs, Statistical Modeling, GCP
Education: B.E. Computer Science
Company: Mindtree
Experience Title: Backend Engineer
Experience Description: Im

In [11]:
for result in rankings[:20]:
    print(
        result["candidate_id"],
        round(result["evidence_score"], 3)
    )

CAND_0000031 0.334
CAND_0000082 0.189
CAND_0000083 0.159
CAND_0000097 0.18
CAND_0000001 0.172
CAND_0000074 0.178
CAND_0000021 0.18
CAND_0000042 0.191
CAND_0000024 0.175
CAND_0000026 0.178
CAND_0000020 0.173
CAND_0000014 0.16
CAND_0000084 0.178
CAND_0000002 0.176
CAND_0000065 0.166
CAND_0000045 0.167
CAND_0000036 0.162
CAND_0000069 0.18
CAND_0000030 0.164
CAND_0000092 0.178


In [6]:
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

print(
    ranker._score_evidence(
        job,
        candidate
    )
)

retrieval systems         Weight=3.0 Best=0.394
ranking systems           Weight=3.0 Best=0.448
recommendation systems    Weight=3.0 Best=0.501
search systems            Weight=3.0 Best=0.417
embeddings                Weight=2.5 Best=0.243
vector databases          Weight=2.5 Best=0.235
python                    Weight=2.0 Best=0.176
llms                      Weight=1.5 Best=0.203
fine-tuning               Weight=1.0 Best=0.268
bm25                      Weight=2.0 Best=0.222

Final Evidence Score = 0.334
0.33374920994677443


In [6]:
from explanation import ExplanationGenerator

In [9]:
explainer = ExplanationGenerator()

In [16]:
for i, result in enumerate(rankings[:5], start=1):

    candidate = result["candidate"]

    print("=" * 80)
    print(f"Rank {i}: {candidate.candidate_id}")

    print(f"Final      : {result['final_score']:.3f}")
    print(f"Evidence   : {result['evidence_score']:.3f}")
    print(f"Skills     : {result['skill_score']:.3f}")
    print(f"Experience : {result['experience_score']:.3f}")
    print(f"Behavior   : {result['behavior_score']:.3f}")

    print("\nDocument:")
    print(candidate.retrieval_document[:800])   # First 800 characters

    print("\nReasoning:")
    print(
        explainer.generate_reasoning(
            job,
            candidate,
            result
        )
    )
    print()

Rank 1: CAND_0000031
Final      : 0.478
Evidence   : 0.309
Skills     : 0.100
Experience : 1.000
Behavior   : 0.793

Document:
Summary: Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.
Headline: Recommendation Systems Engineer | Search, Ranking & Retrieval
Skills: Go,

Reasoning:
Production experience in recommendation systems. Key skills 